# Multilingual BERT Training: English First, Then Arabic

This notebook uses a **two-stage training plan** to avoid the mixed-dataset problem:

1. Train a multilingual **BERT** model on **English GoEmotions**.
2. **Save that checkpoint to Drive**.
3. Continue fine-tuning the saved model on **Arabic ArPanEmo**.
4. **Save the final English + Arabic model under a new name**.

Why this is a good solution:

- it keeps the English checkpoint separately
- it lets us inspect the English model before Arabic fine-tuning
- it reduces the risk of unstable training from mixing two datasets too early
- it matches your request to save after English training and again after the final bilingual stage


## 1. Install and Import Dependencies

If you are running in a fresh Colab runtime, uncomment the install line once.


In [ ]:
# !pip install -q datasets transformers accelerate scikit-learn seaborn

from __future__ import annotations

import json
import re
import shutil
from collections import Counter
from datetime import datetime, timezone
from inspect import signature
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch import nn
from datasets import Dataset, load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

try:
    from google.colab import drive
    COLAB_AVAILABLE = True
except Exception:
    COLAB_AVAILABLE = False

from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)
print('Dependencies imported successfully.')
print('Colab available:', COLAB_AVAILABLE)


## 2. Configuration

We keep the two training stages separate so we can save each one independently.


In [ ]:
BASE_MODEL_NAME = 'bert-base-multilingual-uncased'
ENGLISH_DATASET_NAME = 'mrm8488/goemotions'
ARABIC_DATASET_NAME = 'hanaearg/arpanemo-emotion-dataset-v4-safe-split'

STAGE1_OUTPUT_DIR = Path('models/fine_tuned_bert_english')
FINAL_OUTPUT_DIR = Path('models/fine_tuned_bert_english_arabic')

MAX_LENGTH = 128
MAX_ENGLISH_SAMPLES = None
RANDOM_SEED = 42
TEST_SIZE = 0.2

# Stage 1: English GoEmotions fine-tuning.
ENGLISH_BATCH_SIZE = 4
ENGLISH_EVAL_BATCH_SIZE = 8
ENGLISH_NUM_EPOCHS = 3
ENGLISH_LEARNING_RATE = 2e-5
ENGLISH_WEIGHT_DECAY = 0.01
ENGLISH_WARMUP_RATIO = 0.08
ENGLISH_GRADIENT_ACCUMULATION_STEPS = 2

# Stage 2: Arabic fine-tuning starting from the English checkpoint.
ARABIC_BATCH_SIZE = 4
ARABIC_EVAL_BATCH_SIZE = 8
ARABIC_NUM_EPOCHS = 2
ARABIC_LEARNING_RATE = 1e-5
ARABIC_WEIGHT_DECAY = 0.01
ARABIC_WARMUP_RATIO = 0.08
ARABIC_GRADIENT_ACCUMULATION_STEPS = 2
ARABIC_TRAIN_REPEAT_FACTOR = 3

set_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# GoEmotions raw label columns in the Hugging Face dataset schema.
GOEMOTIONS_RAW_LABELS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
    'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love',
    'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse',
    'sadness', 'surprise', 'neutral',
]

# Shared 7-way emotion space used by both datasets.
SHARED_EMOTION_LABELS = ['joy', 'sadness', 'anger', 'fear', 'surprise', 'disgust', 'neutral']
LABEL2ID = {label: idx for idx, label in enumerate(SHARED_EMOTION_LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

GOEMOTIONS_TO_SHARED = {label: 'joy' for label in ['admiration', 'amusement', 'approval', 'caring', 'desire', 'excitement', 'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief']}
GOEMOTIONS_TO_SHARED.update({label: 'sadness' for label in ['disappointment', 'embarrassment', 'grief', 'remorse', 'sadness']})
GOEMOTIONS_TO_SHARED.update({label: 'anger' for label in ['anger', 'annoyance', 'disapproval']})
GOEMOTIONS_TO_SHARED.update({label: 'fear' for label in ['fear', 'nervousness']})
GOEMOTIONS_TO_SHARED.update({label: 'surprise' for label in ['confusion', 'curiosity', 'realization', 'surprise']})
GOEMOTIONS_TO_SHARED.update({label: 'disgust' for label in ['disgust']})
GOEMOTIONS_TO_SHARED.update({label: 'neutral' for label in ['neutral']})

ARABIC_TO_SHARED = {
    'happiness': 'joy',
    'sadness': 'sadness',
    'anger': 'anger',
    'fear': 'fear',
    'surprise': 'surprise',
    'disgust': 'disgust',
    'neutral': 'neutral',
}

COARSE_LABEL_PRIORITY = ['anger', 'disgust', 'fear', 'sadness', 'surprise', 'joy', 'neutral']
POSITIVE_EMOTIONS = ['joy']
NEGATIVE_EMOTIONS = ['sadness', 'anger', 'fear', 'disgust']
NEUTRAL_EMOTIONS = ['surprise', 'neutral']

print('Device detected:', DEVICE)
print('Base model:', BASE_MODEL_NAME)
print('Stage 1 output:', STAGE1_OUTPUT_DIR.resolve())
print('Final output:', FINAL_OUTPUT_DIR.resolve())
print('Shared label space:', SHARED_EMOTION_LABELS)


## 3. Load Both Datasets

We inspect the English and Arabic datasets before building the training splits.


In [ ]:
english_ds = load_dataset(ENGLISH_DATASET_NAME, split='train')
arabic_ds = load_dataset(ARABIC_DATASET_NAME)

print('English dataset:')
print(english_ds)
print('Arabic dataset:')
print(arabic_ds)

print('English sample rows:')
display(english_ds.select(range(3)).to_pandas())

print('Arabic sample rows from train split:')
display(arabic_ds['train'].select(range(3)).to_pandas())

print('Arabic split sizes:')
for split_name in arabic_ds.keys():
    print('-', split_name, len(arabic_ds[split_name]))


## 4. Helper Functions

We keep the cleaning light and only harmonize the labels into the shared 7-class emotion space.


In [ ]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
WHITESPACE_RE = re.compile(r'\s+')


def clean_text(text):
    if text is None:
        return ''
    text = str(text)
    text = URL_RE.sub(' ', text)
    text = WHITESPACE_RE.sub(' ', text)
    return text.strip()


def map_goemotions_row_to_shared(row):
    positive_positions = np.flatnonzero(row)
    if len(positive_positions) == 0:
        return 'neutral'

    mapped_labels = [GOEMOTIONS_TO_SHARED[GOEMOTIONS_RAW_LABELS[int(pos)]] for pos in positive_positions]
    counts = Counter(mapped_labels)
    max_count = max(counts.values())
    candidates = [label for label, count in counts.items() if count == max_count]

    for label in COARSE_LABEL_PRIORITY:
        if label in candidates:
            return label
    return candidates[0]


def build_english_dataframe(dataset):
    keep_columns = ['text', 'example_very_unclear'] + GOEMOTIONS_RAW_LABELS
    drop_columns = [col for col in dataset.column_names if col not in keep_columns]
    df = dataset.remove_columns(drop_columns).to_pandas()
    df = df[df['example_very_unclear'] == False].copy()  # noqa: E712

    if MAX_ENGLISH_SAMPLES is not None and len(df) > MAX_ENGLISH_SAMPLES:
        df = df.sample(n=MAX_ENGLISH_SAMPLES, random_state=RANDOM_SEED).copy()

    label_matrix = df[GOEMOTIONS_RAW_LABELS].to_numpy(dtype=np.int8)
    shared_labels = [map_goemotions_row_to_shared(row) for row in label_matrix]

    df['label_name'] = shared_labels
    df['label_id'] = df['label_name'].map(LABEL2ID)
    df['language'] = 'en'
    df['source_dataset'] = ENGLISH_DATASET_NAME
    df['text'] = df['text'].map(clean_text)
    return df[['text', 'label_name', 'label_id', 'language', 'source_dataset']].reset_index(drop=True)


def build_arabic_dataframe(dataset_dict):
    frames = []
    for split_name in dataset_dict.keys():
        split_df = dataset_dict[split_name].to_pandas()
        split_df['label_name'] = split_df['output'].map(ARABIC_TO_SHARED)
        split_df['label_id'] = split_df['label_name'].map(LABEL2ID)
        split_df['language'] = 'ar'
        split_df['source_dataset'] = ARABIC_DATASET_NAME
        split_df['split_name'] = split_name
        split_df['text'] = split_df['input'].map(clean_text)
        split_df = split_df.dropna(subset=['label_name'])
        frames.append(split_df[['text', 'label_name', 'label_id', 'language', 'source_dataset', 'split_name']])

    return pd.concat(frames, ignore_index=True)


def sentiment_from_emotion(label_name):
    if label_name in POSITIVE_EMOTIONS:
        return 'positive'
    if label_name in NEGATIVE_EMOTIONS:
        return 'negative'
    return 'neutral'


def table_counts(series):
    return series.value_counts().rename_axis(series.name).reset_index(name='count')


def safe_split(frame, random_state=RANDOM_SEED, test_size=TEST_SIZE, stratify_col='label_name'):
    try:
        train_df, temp_df = train_test_split(
            frame,
            test_size=test_size,
            random_state=random_state,
            stratify=frame[stratify_col],
        )
        val_df, test_df = train_test_split(
            temp_df,
            test_size=0.5,
            random_state=random_state,
            stratify=temp_df[stratify_col],
        )
    except ValueError:
        train_df, temp_df = train_test_split(
            frame,
            test_size=test_size,
            random_state=random_state,
            stratify=None,
        )
        val_df, test_df = train_test_split(
            temp_df,
            test_size=0.5,
            random_state=random_state,
            stratify=None,
        )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def build_class_weights(label_series, num_labels):
    counts = label_series.value_counts().reindex(range(num_labels), fill_value=1).sort_index()
    weights = counts.sum() / (num_labels * counts.to_numpy())
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32), counts


def to_hf_dataset(frame):
    return Dataset.from_pandas(frame[['text', 'label_id']].rename(columns={'label_id': 'labels'}), preserve_index=False)


def tokenize_dataset(dataset, tokenizer):
    return dataset.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=['text'],
    )


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='macro',
        zero_division=0,
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='weighted',
        zero_division=0,
    )
    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
    }


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        self.class_weights = class_weights
        super().__init__(*args, **kwargs)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def make_trainer(model, tokenizer, train_dataset, eval_dataset, output_dir, learning_rate, batch_size, eval_batch_size,
                 num_epochs, warmup_ratio, gradient_accumulation_steps, class_weights):
    training_args_kwargs = {
        'output_dir': str(output_dir),
        'num_train_epochs': num_epochs,
        'per_device_train_batch_size': batch_size,
        'per_device_eval_batch_size': eval_batch_size,
        'gradient_accumulation_steps': gradient_accumulation_steps,
        'learning_rate': learning_rate,
        'weight_decay': 0.01,
        'warmup_ratio': warmup_ratio,
        'save_strategy': 'epoch',
        'save_total_limit': 1,
        'load_best_model_at_end': True,
        'metric_for_best_model': 'f1_macro',
        'greater_is_better': True,
        'logging_steps': 50,
        'dataloader_num_workers': 2,
        'dataloader_pin_memory': torch.cuda.is_available(),
        'fp16': torch.cuda.is_available(),
        'report_to': 'none',
        'seed': RANDOM_SEED,
    }

    training_args_params = TrainingArguments.__init__.__code__.co_varnames
    if 'eval_strategy' in training_args_params:
        training_args_kwargs['eval_strategy'] = 'epoch'
    elif 'evaluation_strategy' in training_args_params:
        training_args_kwargs['evaluation_strategy'] = 'epoch'

    training_args = TrainingArguments(**training_args_kwargs)

    trainer_kwargs = {
        'model': model,
        'args': training_args,
        'train_dataset': train_dataset,
        'eval_dataset': eval_dataset,
        'data_collator': DataCollatorWithPadding(tokenizer=tokenizer),
        'compute_metrics': compute_metrics,
        'class_weights': class_weights,
    }

    trainer_init_params = signature(Trainer.__init__).parameters
    if 'processing_class' in trainer_init_params:
        trainer_kwargs['processing_class'] = tokenizer
    elif 'tokenizer' in trainer_init_params:
        trainer_kwargs['tokenizer'] = tokenizer

    return WeightedTrainer(**trainer_kwargs)


def save_model_bundle(model, tokenizer, output_dir, metadata):
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    metadata_path = output_dir / 'model_metadata.json'
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    return metadata_path


def copy_to_drive(src_dir, drive_name):
    if not COLAB_AVAILABLE:
        print(f'Skipping Drive copy for {src_dir} because this is not Colab.')
        return None

    drive.mount('/content/drive')
    drive_dir = Path('/content/drive/MyDrive') / drive_name
    drive_dir.mkdir(parents=True, exist_ok=True)

    for item in src_dir.iterdir():
        if item.is_file():
            shutil.copy2(item, drive_dir / item.name)

    print('Copied to Drive:', drive_dir)
    return drive_dir


## 5. Build the English and Arabic DataFrames

We keep the two corpora separate so stage 1 and stage 2 can be saved independently.


In [ ]:
english_df = build_english_dataframe(english_ds)
arabic_df = build_arabic_dataframe(arabic_ds)

print('English rows:', len(english_df))
print('Arabic rows:', len(arabic_df))

print('English label distribution:')
display(table_counts(english_df['label_name']))

print('Arabic label distribution:')
display(table_counts(arabic_df['label_name']))

print('Arabic split distribution:')
display(table_counts(arabic_df['split_name']))

print('English sample rows:')
display(english_df.head(5))

print('Arabic sample rows:')
display(arabic_df.head(5))


## 6. Stage 1 ? Train on English GoEmotions and Save to Drive

This is the checkpoint you wanted saved after English training.


In [ ]:
english_train_df, english_val_df, english_test_df = safe_split(english_df)

print('English train rows:', len(english_train_df))
print('English validation rows:', len(english_val_df))
print('English test rows:', len(english_test_df))

print('English train label distribution:')
display(table_counts(english_train_df['label_name']))

print('English validation label distribution:')
display(table_counts(english_val_df['label_name']))

print('English test label distribution:')
display(table_counts(english_test_df['label_name']))

english_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
english_config = AutoConfig.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=len(SHARED_EMOTION_LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

english_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    config=english_config,
    ignore_mismatched_sizes=True,
)
english_model.config.problem_type = 'single_label_classification'
english_model.to(DEVICE)

english_train_hf = to_hf_dataset(english_train_df)
english_val_hf = to_hf_dataset(english_val_df)
english_test_hf = to_hf_dataset(english_test_df)

english_tokenized_train = tokenize_dataset(english_train_hf, english_tokenizer)
english_tokenized_val = tokenize_dataset(english_val_hf, english_tokenizer)
english_tokenized_test = tokenize_dataset(english_test_hf, english_tokenizer)

en_gl_weights, en_counts = build_class_weights(english_train_df['label_id'], len(SHARED_EMOTION_LABELS))
print('English class weights:')
display(pd.DataFrame({'label': SHARED_EMOTION_LABELS, 'count': en_counts.to_numpy(), 'weight': en_gl_weights.numpy()}))

en_gl_trainer = make_trainer(
    model=english_model,
    tokenizer=english_tokenizer,
    train_dataset=english_tokenized_train,
    eval_dataset=english_tokenized_val,
    output_dir=STAGE1_OUTPUT_DIR,
    learning_rate=ENGLISH_LEARNING_RATE,
    batch_size=ENGLISH_BATCH_SIZE,
    eval_batch_size=ENGLISH_EVAL_BATCH_SIZE,
    num_epochs=ENGLISH_NUM_EPOCHS,
    warmup_ratio=ENGLISH_WARMUP_RATIO,
    gradient_accumulation_steps=ENGLISH_GRADIENT_ACCUMULATION_STEPS,
    class_weights=en_gl_weights,
)

print('Starting English training...')
english_train_result = en_gl_trainer.train()
print('English training finished.')
print(english_train_result)

english_val_metrics = en_gl_trainer.evaluate(english_tokenized_val)
english_test_output = en_gl_trainer.predict(english_tokenized_test)
english_test_labels = english_test_hf['labels']
english_test_preds = np.argmax(english_test_output.predictions, axis=-1)
english_test_precision, english_test_recall, english_test_f1, _ = precision_recall_fscore_support(
    english_test_labels,
    english_test_preds,
    average='macro',
    zero_division=0,
)

print('English validation metrics:')
print(english_val_metrics)
print('English test metrics:')
print({
    'accuracy': accuracy_score(english_test_labels, english_test_preds),
    'precision_macro': english_test_precision,
    'recall_macro': english_test_recall,
    'f1_macro': english_test_f1,
})
print('English classification report:')
print(classification_report(english_test_labels, english_test_preds, target_names=SHARED_EMOTION_LABELS, zero_division=0))

stage1_metadata = {
    'model_name': 'fine_tuned_bert_english',
    'stage': 'english_only',
    'base_model': BASE_MODEL_NAME,
    'english_dataset': ENGLISH_DATASET_NAME,
    'label_space': SHARED_EMOTION_LABELS,
    'training_config': {
        'max_length': MAX_LENGTH,
        'batch_size': ENGLISH_BATCH_SIZE,
        'eval_batch_size': ENGLISH_EVAL_BATCH_SIZE,
        'num_epochs': ENGLISH_NUM_EPOCHS,
        'learning_rate': ENGLISH_LEARNING_RATE,
        'weight_decay': ENGLISH_WEIGHT_DECAY,
        'warmup_ratio': ENGLISH_WARMUP_RATIO,
    },
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'output_format': {
        'sentiment': 'positive | negative | neutral',
        'confidence': 'max softmax probability',
        'primary_emotion': 'predicted shared emotion label',
        'emotion_scores': 'top emotion probabilities',
    },
}

stage1_metadata_path = save_model_bundle(english_model, english_tokenizer, STAGE1_OUTPUT_DIR, stage1_metadata)
print('Stage 1 saved to:', STAGE1_OUTPUT_DIR.resolve())
print('Stage 1 metadata saved to:', stage1_metadata_path.resolve())
copy_to_drive(STAGE1_OUTPUT_DIR, 'fine_tuned_bert_english')


## 7. Stage 2 ? Continue Training on Arabic and Save Under a New Name

This stage starts from the saved English checkpoint and fine-tunes it on Arabic ArPanEmo.


In [ ]:
arabic_train_df = arabic_df[arabic_df['split_name'] == 'train'].copy().reset_index(drop=True)
arabic_val_df = arabic_df[arabic_df['split_name'] == 'validation'].copy().reset_index(drop=True)
arabic_test_df = arabic_df[arabic_df['split_name'] == 'test'].copy().reset_index(drop=True)

if ARABIC_TRAIN_REPEAT_FACTOR > 1:
    arabic_train_repeat = pd.concat([arabic_train_df] + [arabic_train_df] * (ARABIC_TRAIN_REPEAT_FACTOR - 1), ignore_index=True)
    arabic_train_df = arabic_train_repeat.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print('Arabic train rows:', len(arabic_train_df))
print('Arabic validation rows:', len(arabic_val_df))
print('Arabic test rows:', len(arabic_test_df))

print('Arabic train label distribution:')
display(table_counts(arabic_train_df['label_name']))

print('Arabic validation label distribution:')
display(table_counts(arabic_val_df['label_name']))

print('Arabic test label distribution:')
display(table_counts(arabic_test_df['label_name']))

# Load the English checkpoint and continue training from it.
final_tokenizer = AutoTokenizer.from_pretrained(STAGE1_OUTPUT_DIR, use_fast=True)
final_config = AutoConfig.from_pretrained(STAGE1_OUTPUT_DIR)
final_model = AutoModelForSequenceClassification.from_pretrained(STAGE1_OUTPUT_DIR, config=final_config)
final_model.config.problem_type = 'single_label_classification'
final_model.to(DEVICE)

arabic_train_hf = to_hf_dataset(arabic_train_df)
arabic_val_hf = to_hf_dataset(arabic_val_df)
arabic_test_hf = to_hf_dataset(arabic_test_df)

arabic_tokenized_train = tokenize_dataset(arabic_train_hf, final_tokenizer)
arabic_tokenized_val = tokenize_dataset(arabic_val_hf, final_tokenizer)
arabic_tokenized_test = tokenize_dataset(arabic_test_hf, final_tokenizer)

ar_weights, ar_counts = build_class_weights(arabic_train_df['label_id'], len(SHARED_EMOTION_LABELS))
print('Arabic class weights:')
display(pd.DataFrame({'label': SHARED_EMOTION_LABELS, 'count': ar_counts.to_numpy(), 'weight': ar_weights.numpy()}))

final_trainer = make_trainer(
    model=final_model,
    tokenizer=final_tokenizer,
    train_dataset=arabic_tokenized_train,
    eval_dataset=arabic_tokenized_val,
    output_dir=FINAL_OUTPUT_DIR,
    learning_rate=ARABIC_LEARNING_RATE,
    batch_size=ARABIC_BATCH_SIZE,
    eval_batch_size=ARABIC_EVAL_BATCH_SIZE,
    num_epochs=ARABIC_NUM_EPOCHS,
    warmup_ratio=ARABIC_WARMUP_RATIO,
    gradient_accumulation_steps=ARABIC_GRADIENT_ACCUMULATION_STEPS,
    class_weights=ar_weights,
)

print('Starting Arabic fine-tuning from the English checkpoint...')
arabic_train_result = final_trainer.train()
print('Arabic training finished.')
print(arabic_train_result)

arabic_val_metrics = final_trainer.evaluate(arabic_tokenized_val)
arabic_test_output = final_trainer.predict(arabic_tokenized_test)
arabic_test_labels = arabic_test_hf['labels']
arabic_test_preds = np.argmax(arabic_test_output.predictions, axis=-1)
ar_precision, ar_recall, ar_f1, _ = precision_recall_fscore_support(
    arabic_test_labels,
    arabic_test_preds,
    average='macro',
    zero_division=0,
)

# Optional retention check: evaluate the final model on the English test set too.
final_english_test_output = final_trainer.predict(english_tokenized_test)
final_english_test_preds = np.argmax(final_english_test_output.predictions, axis=-1)
final_en_precision, final_en_recall, final_en_f1, _ = precision_recall_fscore_support(
    english_test_labels,
    final_english_test_preds,
    average='macro',
    zero_division=0,
)

print('Arabic validation metrics:')
print(arabic_val_metrics)
print('Arabic test metrics:')
print({
    'accuracy': accuracy_score(arabic_test_labels, arabic_test_preds),
    'precision_macro': ar_precision,
    'recall_macro': ar_recall,
    'f1_macro': ar_f1,
})
print('Final model performance on English test set after Arabic fine-tuning:')
print({
    'accuracy': accuracy_score(english_test_labels, final_english_test_preds),
    'precision_macro': final_en_precision,
    'recall_macro': final_en_recall,
    'f1_macro': final_en_f1,
})

print('Arabic classification report:')
print(classification_report(arabic_test_labels, arabic_test_preds, target_names=SHARED_EMOTION_LABELS, zero_division=0))

stage2_metadata = {
    'model_name': 'fine_tuned_bert_english_arabic',
    'stage': 'english_then_arabic',
    'base_model': BASE_MODEL_NAME,
    'english_dataset': ENGLISH_DATASET_NAME,
    'arabic_dataset': ARABIC_DATASET_NAME,
    'label_space': SHARED_EMOTION_LABELS,
    'training_config': {
        'max_length': MAX_LENGTH,
        'english_stage': {
            'batch_size': ENGLISH_BATCH_SIZE,
            'eval_batch_size': ENGLISH_EVAL_BATCH_SIZE,
            'num_epochs': ENGLISH_NUM_EPOCHS,
            'learning_rate': ENGLISH_LEARNING_RATE,
        },
        'arabic_stage': {
            'batch_size': ARABIC_BATCH_SIZE,
            'eval_batch_size': ARABIC_EVAL_BATCH_SIZE,
            'num_epochs': ARABIC_NUM_EPOCHS,
            'learning_rate': ARABIC_LEARNING_RATE,
        },
    },
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'output_format': {
        'sentiment': 'positive | negative | neutral',
        'confidence': 'max softmax probability',
        'primary_emotion': 'predicted shared emotion label',
        'emotion_scores': 'top emotion probabilities',
    },
}

stage2_metadata_path = save_model_bundle(final_model, final_tokenizer, FINAL_OUTPUT_DIR, stage2_metadata)
print('Final bilingual model saved to:', FINAL_OUTPUT_DIR.resolve())
print('Final metadata saved to:', stage2_metadata_path.resolve())
copy_to_drive(FINAL_OUTPUT_DIR, 'fine_tuned_bert_english_arabic')


## 8. Example Inference Using the Final Model


In [ ]:
EMOTION_TO_SENTIMENT = {
    'joy': 'positive',
    'sadness': 'negative',
    'anger': 'negative',
    'fear': 'negative',
    'disgust': 'negative',
    'surprise': 'neutral',
    'neutral': 'neutral',
}


def predict_emotion(text):
    final_model.eval()
    cleaned_text = clean_text(text)
    inputs = final_tokenizer(
        cleaned_text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = final_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    top_index = int(np.argmax(probs))
    top_label = ID2LABEL[top_index]
    top_sentiment = EMOTION_TO_SENTIMENT.get(top_label, 'neutral')
    top_indices = np.argsort(probs)[::-1][:5]
    top_emotions = [
        {'label': ID2LABEL[int(idx)], 'score': float(probs[int(idx)])}
        for idx in top_indices
    ]

    return {
        'sentiment': top_sentiment,
        'confidence': float(probs[top_index]),
        'primary_emotion': top_label,
        'emotion_scores': top_emotions,
    }

sample_texts = [
    "I am really happy with this result!",
    "هذا الخبر جميل جدًا وأنا سعيدة به",
    "I am frustrated and disappointed by the delay.",
    "أنا غاضب جدًا من هذا التصرف",
]

for sample in sample_texts:
    result = predict_emotion(sample)
    print('Text:', sample)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print('-')


## 9. Final Summary

This notebook follows the exact workflow you asked for:

- **Stage 1:** train on English GoEmotions
- **Save Stage 1** to Drive with a dedicated folder name
- **Stage 2:** load Stage 1 and continue training on Arabic ArPanEmo
- **Save the final bilingual model** to a new folder name

Saved locations:

- English checkpoint: `models/fine_tuned_bert_english/`
- Final bilingual checkpoint: `models/fine_tuned_bert_english_arabic/`
- Drive copy of English checkpoint: `MyDrive/fine_tuned_bert_english/`
- Drive copy of final checkpoint: `MyDrive/fine_tuned_bert_english_arabic/`

This is usually a cleaner solution than mixing both datasets in one training loop.
